In [1]:
import sys
sys.path.append(r"C:\Users\HP\Desktop\image-search")

In [2]:
from torchvision import datasets, transforms

dataset = datasets.CIFAR10(
    root="data/cifar10",
    download=False,  # already downloaded
    transform=transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor()
    ])
)

# Access like this
image, label = dataset[0]   # image is a tensor (3, 224, 224)
print(f"Dataset size: {len(dataset)}")  # 50000
print(f"Classes: {dataset.classes}")

Dataset size: 50000
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


c:\Users\HP\Desktop\image-search\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [ ]:
import clip
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/16", device=device)
print(f"Using device: {device} and model: ViT-B/32")

Using device: cuda and model: ViT-B/32


In [4]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 4050 Laptop GPU


In [5]:
from torch.utils.data import DataLoader,Subset

subset=Subset(dataset, range(100)) #first 100 for testing
loader = DataLoader(subset, batch_size=16, shuffle=False)

embeddings=[]

model.eval()
with torch.no_grad():
    for images, labels in loader:
        images = images.to(device)
        features=model.encode_image(images)
        features=features/features.norm(dim=-1, keepdim=True) #normalisation
        embeddings.append(features.cpu())

embeddings=torch.cat(embeddings)
print(f"Embeddings shape: {embeddings.shape}")  # (100, 512)
#100-> number of images, 512-> embedding dimension (size of clip embedding vector)

Embeddings shape: torch.Size([100, 512])


FAISS is essentially a collection of nearest neighbour search algorithms optimised for high-dimensional vectors.

The one you're using now, IndexFlatIP, is just brute force — compare query vector against every stored vector, return closest. Simple but slow at scale.
The progression you'll follow:

Stage 1 — IndexFlatIP — brute force, exact, fine for 100–50k vectors
Stage 3 — IndexIVFFlat — clusters vectors into groups, only searches relevant clusters, much faster at 25k+

import faiss
import numpy as np

embeddings_np = embeddings.numpy().astype('float32')  # Convert to numpy array

dimension=512
index=faiss.IndexFlatIP(dimension) #IP for inner product (cosine similarity)
index.add(embeddings_np)

print(f"Total embeddings indexed: {index.ntotal}")  # Should be 100

#first img as query
query=embeddings_np[0:1]  # shape (1, 512)

k=5 #top 5 similar images
distances,indices=index.search(query, k)

print(f"Top {k} similar image indices:", indices[0])
print(f"Similiarity scores:", distances[0])

In [6]:
from torch.utils.data import DataLoader

full_loader=DataLoader(dataset, batch_size=16, shuffle=True)
all_embeds=[]

model.eval()
with torch.no_grad():
    for i,(image,labels) in enumerate(full_loader):
        image=image.to(device)
        features=model.encode_image(image)
        features=features/features.norm(dim=-1, keepdim=True)
        all_embeds.append(features.cpu())

        if i%100==0:
            print(f"Batch {i}/{len(full_loader)} processed")
all_embeds=torch.cat(all_embeds)
print(f"All embeddings shape: {all_embeds.shape}")  # (50000, 512)

Batch 0/3125 processed
Batch 100/3125 processed
Batch 200/3125 processed
Batch 300/3125 processed
Batch 400/3125 processed


KeyboardInterrupt: 

In [ ]:
import faiss 
import numpy as np

all_embeds_np=all_embeds.numpy().astype('float32')

dimension=512   
index=faiss.IndexFlatIP(dimension)  # Inner Product for cosine similarity
index.add(all_embeds_np)  # Add all embeddings to the index

print(f"Total embeddings indexed: {index.ntotal}")  #50000

# save to disk
faiss.write_index(index,"data/index.faiss")
print("Index saved.")



Total embeddings indexed: 50000
Index saved.


In [ ]:
import pandas as pd

labels=[dataset[i][1] for i in range(len(dataset))]
class_names=[dataset.classes[i] for i in labels]

df=pd.DataFrame({"index":range(len(dataset)),"label":labels,"class_name":class_names})
df.to_csv("data/metadata.csv",index=False)
print("Metadata saved.")
print(df.head())

Metadata saved.
   index  label  class_name
0      0      6        frog
1      1      9       truck
2      2      9       truck
3      3      4        deer
4      4      1  automobile


In [ ]:
#verification
index_loaded=faiss.read_index("data/index.faiss")
df_loaded=pd.read_csv("data/metadata.csv")

#query with first image
query=all_embeds_np[0:1]
distances,indices=index_loaded.search(query,k=5)

print("Query image class:", df_loaded.iloc[0]["class_name"]) #iloc->integer location based indexing
print("Top 5 similar images:")
for idx,dist in zip(indices[0],distances[0]):
    print(f"Index: {idx}, Class: {df_loaded.iloc[idx]['class_name']}, Score: {dist:.4f}")



Query image class: frog
Top 5 similar images:
Index: 0, Class: frog, Score: 1.0001
Index: 23161, Class: deer, Score: 0.9725
Index: 45704, Class: automobile, Score: 0.9651
Index: 34383, Class: dog, Score: 0.9621
Index: 33132, Class: cat, Score: 0.9613
